In [18]:
# [MISSION 1] Web Server Access Log Analysis
from pathlib import Path
import csv
import json

# Student-specific parameters
STUDENT_ID = "TUPM-26-1138"
LAST_NAME = "Moya"
SEED_DIGIT = 8
ID_SUM = 23

# Directories
BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / "data"
OUTPUT_DIR = BASE_DIR / "output"
BACKUP_DIR = BASE_DIR / "backup"

for d in [DATA_DIR, OUTPUT_DIR, BACKUP_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Find any CSV file in the workspace or data folder
csv_files = list(BASE_DIR.glob("**/*.csv"))
dataset_path = None
for f in csv_files:
    if f.name == "weblog.csv":
        dataset_path = f
        break

if not dataset_path:
    print("Error: weblog.csv not found. Please place it in your data folder.")
else:
    print(f"Found dataset at: {dataset_path}")

    # Parameters from rules
    records_to_process = 1000 + ID_SUM  # 1023 records
    target_class = str((SEED_DIGIT % 4) + 2)  # '2' for 2xx status codes

    filtered_records = []
    status_counts = {}
    url_counts = {}
    count = 0

    with open(dataset_path, "r", encoding="utf-8", errors="ignore") as f:
        reader = csv.reader(f)
        header = next(reader, None)
        print("Log Fields:", header)
        
        for row in reader:
            if count >= records_to_process:
                break
            if len(row) >= 4:
                ip, time_val, url, status = row[0], row[1], row[2], row[3]
                
                status_counts[status] = status_counts.get(status, 0) + 1
                
                if status.startswith(target_class):
                    filtered_records.append({
                        "ip": ip,
                        "time": time_val,
                        "url": url,
                        "status": status
                    })
                    url_counts[url] = url_counts.get(url, 0) + 1
            count += 1

    # Find top 5 most frequently requested URLs within target class
    top_urls = sorted(url_counts.items(), key=lambda x: x[1], reverse=True)[:5]

    print(f"\nProcessed Records: {count}")
    print(f"Target Status Class: {target_class}xx")
    print(f"Matching Records Found: {len(filtered_records)}")
    print("Top 5 Requested URLs:", top_urls)

    # Export selected records to a filtered CSV file
    filtered_csv_path = OUTPUT_DIR / f"{LAST_NAME}_{STUDENT_ID}_weblog_filtered.csv"
    with open(filtered_csv_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=["ip", "time", "url", "status"])
        writer.writeheader()
        writer.writerows(filtered_records)

    # Write summary to a JSON file
    json_summary_path = OUTPUT_DIR / f"{LAST_NAME}_{STUDENT_ID}_weblog_summary.json"
    summary_data = {
        "student_id": STUDENT_ID,
        "records_processed": count,
        "target_status_class": f"{target_class}xx",
        "matching_records_count": len(filtered_records),
        "status_counts": status_counts,
        "top_5_urls": top_urls
    }

    with open(json_summary_path, "w", encoding="utf-8") as f:
        json.dump(summary_data, f, indent=4)

    # Create a backup of the JSON file and verify
    import shutil
    backup_json_path = BACKUP_DIR / json_summary_path.name
    shutil.copy2(json_summary_path, backup_json_path)

    print("\nFiles successfully generated:")
    print(" - Filtered CSV:", filtered_csv_path.name)
    print(" - JSON Summary:", json_summary_path.name)
    print(" - Backup verified exists:", backup_json_path.exists())

Found dataset at: c:\Users\ADMIN\Lab6_File_Handling_Moya\weblog.csv
Log Fields: ['IP', 'Time', 'URL', 'Staus']

Processed Records: 1023
Target Status Class: 2xx
Matching Records Found: 853
Top 5 Requested URLs: [('GET /login.php HTTP/1.1', 105), ('GET /home.php HTTP/1.1', 91), ('GET /css/bootstrap.min.css HTTP/1.1', 38), ('GET /css/normalize.css HTTP/1.1', 38), ('GET /js/vendor/jquery-1.12.0.min.js HTTP/1.1', 37)]

Files successfully generated:
 - Filtered CSV: Moya_TUPM-26-1138_weblog_filtered.csv
 - JSON Summary: Moya_TUPM-26-1138_weblog_summary.json
 - Backup verified exists: True


In [19]:
# [MISSION 2] MOBILE DEVICE USAGE PROFILING
from pathlib import Path
import csv
import json
import shutil

# Student-specific parameters
STUDENT_ID = "TUPM-26-1138"
LAST_NAME = "Moya"
SEED_DIGIT = 8

# Directories
BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / "data"
OUTPUT_DIR = BASE_DIR / "output"
BACKUP_DIR = BASE_DIR / "backup"

for d in [DATA_DIR, OUTPUT_DIR, BACKUP_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Find user_behavior_dataset.csv
csv_files = list(BASE_DIR.glob("**/*.csv"))
dataset_path = None
for f in csv_files:
    if f.name == "user_behavior_dataset.csv":
        dataset_path = f
        break

if not dataset_path:
    print("Error: user_behavior_dataset.csv not found. Please place it in your data folder.")
else:
    print(f"Found dataset at: {dataset_path}")

    # Rules calculation
    target_os = "Android" if SEED_DIGIT % 2 == 0 else "iOS"
    target_behavior_class = (SEED_DIGIT % 3) + 1

    total_records = 0
    os_matches = 0
    filtered_users = []
    
    total_battery_drain = 0.0
    total_data_usage = 0.0

    with open(dataset_path, "r", encoding="utf-8", errors="ignore") as f:
        reader = csv.reader(f)
        header = next(reader, None)
        print("Dataset Fields:", header)
        
        for row in reader:
            total_records += 1
            if len(row) >= 6:
                # Adjust column indexing based on actual header fields if needed
                # Expected fields typically: User ID, Device Model, Operating System, App Usage Time (min/day), Screen On Time (hours/day), Battery Drain (mAh/day), Number of Apps Installed, Data Usage (MB/day), Age, Gender, User Behavior Class
                user_id = row[0]
                device_model = row[1]
                os = row[2]
                battery_drain = row[5] if len(row) > 5 else "0"
                data_usage = row[7] if len(row) > 7 else "0"
                behavior_class = row[10] if len(row) > 10 else "0"

                if os.strip().lower() == target_os.lower():
                    os_matches += 1
                    try:
                        if int(behavior_class.strip()) == target_behavior_class:
                            filtered_users.append({
                                "user_id": user_id,
                                "device_model": device_model,
                                "operating_system": os,
                                "battery_drain": float(battery_drain),
                                "data_usage": float(data_usage)
                            })
                            total_battery_drain += float(battery_drain)
                            total_data_usage += float(data_usage)
                    except ValueError:
                        continue

    # Calculate averages
    matching_count = len(filtered_users)
    avg_battery = total_battery_drain / matching_count if matching_count > 0 else 0
    avg_data = total_data_usage / matching_count if matching_count > 0 else 0

    print(f"\nTotal Records in Dataset: {total_records}")
    print(f"Records Matching OS ({target_os}): {os_matches}")
    print(f"Records Matching Behavior Class ({target_behavior_class}): {matching_count}")
    print(f"Average Battery Drain: {avg_battery:.2f} mAh/day")
    print(f"Average Data Usage: {avg_data:.2f} MB/day")

    # Export selected records to CSV
    exported_csv_path = OUTPUT_DIR / f"{LAST_NAME}_{STUDENT_ID}_device_filtered.csv"
    with open(exported_csv_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=["user_id", "device_model", "operating_system", "battery_drain", "data_usage"])
        writer.writeheader()
        writer.writerows(filtered_users)

    # Export JSON Summary
json_summary_path = OUTPUT_DIR / f"{LAST_NAME}_{STUDENT_ID}_device_summary.json"
summary_data = {
    "student_id": STUDENT_ID,
    "target_os": target_os,
    "target_behavior_class": target_behavior_class,
    "total_records": total_records,
    "os_matches": os_matches,
    "matching_record_count": matching_count,
    "average_battery_drain_mAh_day": round(avg_battery, 2),
    "average_data_usage_MB_day": round(avg_data, 2)
}

with open(json_summary_path, "w", encoding="utf-8") as f:
    json.dump(summary_data, f, indent=4)

# Create backup and check file size
backup_csv_path = BACKUP_DIR / exported_csv_path.name
shutil.copy2(exported_csv_path, backup_csv_path)
backup_size = backup_csv_path.stat().st_size

print(f"\nExported CSV: {exported_csv_path.name}")
print(f"JSON Summary: {json_summary_path.name}")
print(f"Backup File Size: {backup_size} bytes (Verified: {backup_csv_path.exists()})")

Found dataset at: c:\Users\ADMIN\Lab6_File_Handling_Moya\Lab6_File_Handling_Moya\data\user_behavior_dataset.csv
Dataset Fields: ['User ID', 'Device Model', 'Operating System', 'App Usage Time (min/day)', 'Screen On Time (hours/day)', 'Battery Drain (mAh/day)', 'Number of Apps Installed', 'Data Usage (MB/day)', 'Age', 'Gender', 'User Behavior Class']

Total Records in Dataset: 700
Records Matching OS (Android): 554
Records Matching Behavior Class (3): 111
Average Battery Drain: 1521.69 mAh/day
Average Data Usage: 820.26 MB/day

Exported CSV: Moya_TUPM-26-1138_device_filtered.csv
JSON Summary: Moya_TUPM-26-1138_device_summary.json
Backup File Size: 4493 bytes (Verified: True)


In [20]:
# [EXERCISE 3 FIXED] CYBERSECURITY INCIDENT REPORT
from pathlib import Path
import csv
import json
import shutil

# Student-specific parameters
STUDENT_ID = "TUPM-26-1138"
LAST_NAME = "Moya"
SEED_DIGIT = 8
TARGET_YEAR = 2015 + SEED_DIGIT  # 2023

# Directories
BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / "data"
OUTPUT_DIR = BASE_DIR / "output"
BACKUP_DIR = BASE_DIR / "backup"

for d in [DATA_DIR, OUTPUT_DIR, BACKUP_DIR]:
    d.mkdir(parents=True, exist_ok=True)

csv_files = list(BASE_DIR.glob("**/*.csv"))
dataset_path = None
for f in csv_files:
    if "global" in f.name.lower() or "cybersecurity" in f.name.lower():
        dataset_path = f
        break

total_records = 0
target_incidents = []
attack_counts = {}
parse_success = 0
total_loss = 0.0

with open(dataset_path, "r", encoding="utf-8", errors="ignore") as f:
    reader = csv.reader(f)
    header = next(reader, None)
    print("Dataset Fields:", header)
    
    for row in reader:
        total_records += 1
        if len(row) >= 5:
            country = row[0]
            year_val = row[1]
            attack_type = row[2]
            loss_val = row[4]  # Financial Loss (in Million $) is at index 4 based on your header

            try:
                # Safely parse year and financial loss
                clean_year = int(float(year_val.strip()))
                if clean_year == TARGET_YEAR:
                    loss = float(loss_val.strip()) if loss_val.strip() else 0.0
                    target_incidents.append({
                        "row": row,
                        "loss": loss,
                        "attack_type": attack_type
                    })
                    attack_counts[attack_type] = attack_counts.get(attack_type, 0) + 1
                    total_loss += loss
            except (ValueError, TypeError):
                continue

incident_count = len(target_incidents)
avg_loss = total_loss / incident_count if incident_count > 0 else 0

# Sort to find top 5 costliest incidents
top_costliest = sorted(target_incidents, key=lambda x: x["loss"], reverse=True)[:5]

print(f"\nTotal Records in Dataset: {total_records}")
print(f"Incidents in Target Year ({TARGET_YEAR}): {incident_count}")
print(f"Total Financial Loss: ${total_loss:.2f} Million")
print(f"Average Financial Loss: ${avg_loss:.2f} Million")

# Export filtered CSV
filtered_csv_path = OUTPUT_DIR / f"{LAST_NAME}_{STUDENT_ID}_cyber_filtered.csv"
with open(filtered_csv_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    if header:
        writer.writerow(header)
    for inc in target_incidents:
        writer.writerow(inc["row"])

# Export JSON Summary
json_summary_path = OUTPUT_DIR / f"{LAST_NAME}_{STUDENT_ID}_cyber_summary.json"
summary_data = {
    "student_id": STUDENT_ID,
    "target_year": TARGET_YEAR,
    "total_records": total_records,
    "incidents_in_target_year": incident_count,
    "total_financial_loss_million": round(total_loss, 2),
    "average_financial_loss_million": round(avg_loss, 2),
    "attack_type_counts": attack_counts,
    "top_5_costliest_incidents": [
        {"loss_million": inc["loss"], "attack_type": inc["attack_type"]} 
        for inc in top_costliest
    ]
}

with open(json_summary_path, "w", encoding="utf-8") as f:
    json.dump(summary_data, f, indent=4)

# Generate Text Report
report_path = OUTPUT_DIR / f"{LAST_NAME}_{STUDENT_ID}_cyber_report.txt"
with open(report_path, "w", encoding="utf-8") as f:
    f.write(f"CYBERSECURITY INCIDENT REPORT ({TARGET_YEAR})\n")
    f.write(f"Student: {LAST_NAME} ({STUDENT_ID})\n")
    f.write("="*50 + "\n\n")
    f.write(f"Total Dataset Records: {total_records}\n")
    f.write(f"Incidents in {TARGET_YEAR}: {incident_count}\n")
    f.write(f"Total Financial Loss: ${total_loss:,.2f} Million\n")
    f.write(f"Average Financial Loss: ${avg_loss:,.2f} Million\n\n")
    f.write("Attack Type Breakdown:\n")
    for atk, cnt in attack_counts.items():
        f.write(f" - {atk}: {cnt}\n")
    f.write("\nTop 5 Costliest Incidents:\n")
    for i, inc in enumerate(top_costliest, 1):
        f.write(f" {i}. Loss: ${inc['loss']:,.2f}M | Type: {inc['attack_type']}\n")

# Create Backup of JSON file and verify
backup_json_path = BACKUP_DIR / json_summary_path.name
shutil.copy2(json_summary_path, backup_json_path)

print(f"\nFiles successfully generated:")
print(f" - Filtered CSV: {filtered_csv_path.name}")
print(f" - JSON Summary: {json_summary_path.name}")
print(f" - Text Report: {report_path.name}")
print(f" - Backup Verified: {backup_json_path.exists()}")

Dataset Fields: ['Country', 'Year', 'Attack Type', 'Target Industry', 'Financial Loss (in Million $)', 'Number of Affected Users', 'Attack Source', 'Security Vulnerability Type', 'Defense Mechanism Used', 'Incident Resolution Time (in Hours)']

Total Records in Dataset: 3000
Incidents in Target Year (2023): 315
Total Financial Loss: $15958.08 Million
Average Financial Loss: $50.66 Million

Files successfully generated:
 - Filtered CSV: Moya_TUPM-26-1138_cyber_filtered.csv
 - JSON Summary: Moya_TUPM-26-1138_cyber_summary.json
 - Text Report: Moya_TUPM-26-1138_cyber_report.txt
 - Backup Verified: True
